# Walmart Weekly Sales — Linear Regression

This notebook builds a linear regression model to predict **weekly sales** across 45 Walmart stores using the [Walmart Dataset](https://www.kaggle.com/datasets/yasserh/walmart-dataset) from Kaggle.

**Pipeline overview:**
1. Load & inspect data
2. Feature engineering (date decomposition, encoding)
3. Statistical analysis — identify which features are significantly associated with sales
4. Feature selection based on significance tests
5. Train / test split
6. Train a Linear Regression model
7. Evaluate with R², MAE, and RMSE
8. Interpret coefficients and diagnostic plots

## 1. Setup

Install `kagglehub` to download the dataset directly from Kaggle. You will need a Kaggle account and API key configured, **or** you can upload `Walmart.csv` manually and adjust the `file_path` variable below.

In [ ]:
!pip install kagglehub -q

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. Load Data

In [ ]:
import kagglehub

path = kagglehub.dataset_download('yasserh/walmart-dataset')
file_path = os.path.join(path, os.listdir(path)[0])

# --- Alternative: upload Walmart.csv manually and set ---
# file_path = 'Walmart.csv'

df = pd.read_csv(file_path)
print(f'Shape: {df.shape}')
df.head()

## 3. Feature Engineering

The `Date` column is converted to datetime and decomposed into `Week`, `Month`, and `Year` — these capture seasonality effects. `Holiday_Flag` is label-encoded (it is already binary, so this is a no-op, but kept for consistency).

In [ ]:
df['Date']  = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Week']  = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year
df.drop(columns=['Date'], inplace=True)

le = LabelEncoder()
df['Holiday_Flag'] = le.fit_transform(df['Holiday_Flag'])

print(df.dtypes)
df.describe()

## 4. Statistical Analysis

Before modelling, we test each feature's association with `Weekly_Sales` using the appropriate test for its measurement level:

| Variable type | Test | Why |
|---|---|---|
| Continuous (`Temperature`, `Fuel_Price`, `CPI`, `Unemployment`) | Pearson r | Measures linear correlation between two continuous variables |
| Ordinal (`Week`, `Month`) | Kendall's τ | Non-parametric rank correlation — robust to non-normality and ties |
| Binary (`Holiday_Flag`) | Two-sample t-test | Compares means of two groups; Welch's variant used if variances differ (Levene's test) |
| Categorical (`Store`) | One-way ANOVA + Kruskal-Wallis | ANOVA tests equality of means across groups; Kruskal-Wallis is the non-parametric fallback when residuals are non-normal |

Features with **p < 0.05** are retained for modelling.

In [ ]:
target      = 'Weekly_Sales'
continuous  = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
ordinal     = ['Week', 'Month']
binary      = ['Holiday_Flag']
categorical = ['Store']

### 4a. Pearson Correlation — Continuous Variables

In [ ]:
print('=' * 55)
print('PEARSON CORRELATION — Continuous Variables')
print('=' * 55)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Weekly Sales vs Continuous Variables (Pearson)', fontsize=13, fontweight='bold')

for ax, col in zip(axes.flat, continuous):
    r, p = stats.pearsonr(df[col], df[target])
    print(f'  {col:<20} r = {r:+.4f}  p = {p:.4f}  {"*" if p < 0.05 else ""}')
    sns.regplot(data=df, x=col, y=target, ax=ax,
                scatter_kws={'alpha': 0.3, 's': 10},
                line_kws={'color': 'crimson', 'linewidth': 2})
    ax.set_title(f'{col}  |  r = {r:+.3f}, p = {p:.4f}', fontsize=10)

plt.tight_layout()
plt.show()

### 4b. Kendall's τ — Ordinal Variables

In [ ]:
print('=' * 55)
print("KENDALL'S TAU — Ordinal Variables")
print('=' * 55)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Weekly Sales vs Ordinal Variables (Kendall)', fontsize=13, fontweight='bold')

for ax, col in zip(axes, ordinal):
    tau, p = stats.kendalltau(df[col], df[target])
    print(f'  {col:<20} tau = {tau:+.4f}  p = {p:.4f}  {"*" if p < 0.05 else ""}')
    mean_sales = df.groupby(col)[target].mean().reset_index()
    ax.scatter(df[col], df[target], alpha=0.2, s=8, color='steelblue')
    ax.plot(mean_sales[col], mean_sales[target], color='crimson', linewidth=2, marker='o', markersize=4)
    ax.set_title(f'{col}  |  tau = {tau:+.3f}, p = {p:.4f}', fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Weekly Sales')

plt.tight_layout()
plt.show()

### 4c. Two-Sample t-Test — Holiday Flag

In [ ]:
print('=' * 55)
print('TWO-SAMPLE T-TEST — Holiday_Flag')
print('=' * 55)

group0 = df[df['Holiday_Flag'] == 0][target]
group1 = df[df['Holiday_Flag'] == 1][target]

lev_stat, lev_p = stats.levene(group0, group1)
equal_var = lev_p > 0.05
t_stat, p_ttest = stats.ttest_ind(group0, group1, equal_var=equal_var)

print(f"  Levene's test: p = {lev_p:.4f} -> {'equal variances' if equal_var else 'unequal variances (Welch)'}")
print(f'  t = {t_stat:.4f},  p = {p_ttest:.4f}  {"*" if p_ttest < 0.05 else ""}')
print(f'  Mean non-holiday: ${group0.mean():,.0f}  |  Mean holiday: ${group1.mean():,.0f}')

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='Holiday_Flag', y=target, palette=['#5B9BD5', '#E15759'], ax=ax)
ax.set_xticklabels(['Non-Holiday (0)', 'Holiday (1)'])
ax.set_title(f'Weekly Sales by Holiday Flag\nt = {t_stat:.3f}, p = {p_ttest:.4f}', fontweight='bold')
ax.set_ylabel('Weekly Sales')
plt.tight_layout()
plt.show()

### 4d. ANOVA + Kruskal-Wallis — Store

In [ ]:
print('=' * 55)
print('ANOVA + KRUSKAL-WALLIS — Store')
print('=' * 55)

groups = [g[target].values for _, g in df.groupby('Store')]
f_stat, p_anova = stats.f_oneway(*groups)

grand_mean = df[target].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in [pd.Series(g) for g in groups])
ss_total   = sum((df[target] - grand_mean) ** 2)
eta_sq     = ss_between / ss_total

print(f'  ANOVA:          F = {f_stat:.4f},  p = {p_anova:.4f}  {"*" if p_anova < 0.05 else ""}')
print(f"  Effect size eta2 = {eta_sq:.4f} ({'large' if eta_sq > 0.14 else 'medium' if eta_sq > 0.06 else 'small'})")

residuals = df[target] - df.groupby('Store')[target].transform('mean')
_, p_shapiro = stats.shapiro(residuals.sample(500, random_state=42))
print(f'  Shapiro-Wilk on residuals: p = {p_shapiro:.4f}')

if p_shapiro < 0.05:
    h_stat, p_kruskal = stats.kruskal(*groups)
    print(f'  -> Non-normal residuals: reporting Kruskal-Wallis')
    print(f'  Kruskal-Wallis: H = {h_stat:.4f},  p = {p_kruskal:.4f}  {"*" if p_kruskal < 0.05 else ""}')

store_order = df.groupby('Store')[target].median().sort_values().index
fig, ax = plt.subplots(figsize=(20, 5))
sns.boxplot(data=df, x='Store', y=target, order=store_order, palette='Blues', ax=ax)
ax.set_title(f'Weekly Sales by Store (sorted by median)\nANOVA F = {f_stat:.2f}, p = {p_anova:.4f}, eta2 = {eta_sq:.3f}', fontweight='bold')
ax.tick_params(axis='x', labelsize=7)
plt.tight_layout()
plt.show()

### 4e. Summary Table

In [ ]:
print('=' * 55)
print('SUMMARY')
print('=' * 55)
print(f'{"Variable":<20} {"Test":<22} {"Statistic":>12}  {"p-value":>10}  Sig.')
print('-' * 70)
for col in continuous:
    r, p = stats.pearsonr(df[col], df[target])
    print(f'{col:<20} {"Pearson r":<22} {r:>+12.4f}  {p:>10.4f}  {"*" if p < 0.05 else ""}')
for col in ordinal:
    tau, p = stats.kendalltau(df[col], df[target])
    print(f'{col:<20} {"Kendall tau":<22} {tau:>+12.4f}  {p:>10.4f}  {"*" if p < 0.05 else ""}')
print(f'{"Holiday_Flag":<20} {"Two-sample t-test":<22} {t_stat:>+12.4f}  {p_ttest:>10.4f}  {"*" if p_ttest < 0.05 else ""}')
print(f'{"Store":<20} {"ANOVA F":<22} {f_stat:>+12.4f}  {p_anova:>10.4f}  {"*" if p_anova < 0.05 else ""}')

## 5. Feature Selection

Only features with **p < 0.05** in their respective tests are retained. `Store` is one-hot encoded since it is a nominal categorical variable — `drop_first=True` drops one dummy to avoid perfect multicollinearity (the dummy trap).

In [ ]:
sig_cols = []

for col in continuous + ordinal:
    if col in continuous:
        _, p = stats.pearsonr(df[col], df[target])
    else:
        _, p = stats.kendalltau(df[col], df[target])
    if p < 0.05:
        sig_cols.append(col)
    print(f'  {col:<20} p = {p:.4f}  {"-> KEEP" if p < 0.05 else "-> DROP"}')

if p_ttest < 0.05:
    sig_cols.append('Holiday_Flag')
print(f'  {"Holiday_Flag":<20} p = {p_ttest:.4f}  {"-> KEEP" if p_ttest < 0.05 else "-> DROP"}')

if p_anova < 0.05:
    sig_cols.append('Store')
print(f'  {"Store":<20} p = {p_anova:.4f}  {"-> KEEP" if p_anova < 0.05 else "-> DROP"}')

print(f'\nRetained features: {sig_cols}')

df_model = df[sig_cols + [target]].copy()

if 'Store' in sig_cols:
    df_model = pd.get_dummies(df_model, columns=['Store'], drop_first=True)

print(f'Shape after encoding: {df_model.shape}')

## 6. Train / Test Split

An 80/20 split with a fixed `random_state` for reproducibility. Stratification is not needed for regression targets.

In [ ]:
X = df_model.drop(columns=[target])
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  |  y_test: {y_test.shape}')

## 7. Train Linear Regression

Features are standardized with `StandardScaler` (mean = 0, std = 1) so that coefficients are on the same scale and directly comparable. The scaler is **fit only on the training set** — we then apply those same training statistics when transforming the test set. Fitting on test data would leak information and give an overly optimistic evaluation.

Ordinary Least Squares (OLS) linear regression then finds the hyperplane that minimizes the sum of squared residuals across all training samples.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('Model trained.')

## 8. Evaluation

| Metric | Formula | Interpretation |
|---|---|---|
| **R²** | 1 - SS_res / SS_tot | Proportion of variance explained. 1.0 = perfect, 0 = no better than predicting the mean. |
| **MAE** | mean(|y - y_hat|) | Average absolute dollar error per prediction. Easy to interpret directly. |
| **RMSE** | sqrt(mean((y - y_hat)²)) | Like MAE but penalises large errors more heavily. RMSE > MAE means some predictions have significant outlier errors. |

Comparing **Train R²** vs **Test R²** is a quick overfitting check — values close together indicate the model generalises well.

In [ ]:
y_pred_train = model.predict(X_train_scaled)
y_pred_test  = model.predict(X_test_scaled)

train_r2  = r2_score(y_train, y_pred_train)
test_r2   = r2_score(y_test,  y_pred_test)
test_mae  = mean_absolute_error(y_test, y_pred_test)
test_rmse = mean_squared_error(y_test, y_pred_test) ** 0.5

print('=' * 40)
print('LINEAR REGRESSION — Results')
print('=' * 40)
print(f'  Train R²:   {train_r2:.4f}')
print(f'  Test  R²:   {test_r2:.4f}')
print(f'  Test  MAE:  ${test_mae:,.0f}')
print(f'  Test  RMSE: ${test_rmse:,.0f}')

## 9. Coefficients

Because features were standardized, each coefficient represents the change in `Weekly_Sales` per **one standard deviation** increase in that feature — magnitudes are directly comparable across all variables.

Store dummy variables are expected to dominate since store identity (size, location, demographics) is far more predictive than week-to-week economic indicators.

In [ ]:
coef_df = pd.DataFrame({
    'Feature':     X_train.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df.to_string(index=False))

## 10. Diagnostic Plots

**Residuals vs Predicted** — checks model assumptions. Ideally a flat, random cloud centred on zero. A funnel shape indicates *heteroscedasticity* (variance grows with prediction size); a curve suggests a non-linear relationship the model has not captured.

**Actual vs Predicted** — each point is one test observation. Points should cluster tightly along the diagonal (perfect prediction line). Scatter away from it represents prediction error.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_test, y_test - y_pred_test, alpha=0.3, s=10)
axes[0].axhline(0, color='crimson', linewidth=1.5)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].scatter(y_test, y_pred_test, alpha=0.3, s=10)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[1].plot(lims, lims, color='crimson', linewidth=1.5)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Actual vs Predicted  (R² = {test_r2:.3f})')

plt.tight_layout()
plt.show()